**Task**: Build a decision tree classifier to predict whether a customer will purchase a product or service based on their demographic and behavioral data. Use a dataset such as the Bank Marketing dataset from the UCI Machine Learning Repository.

In [113]:
import pandas as pd

df = pd.read_csv("bank.csv", sep=";")
df.info()
df["y"].unique()
df["y"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


y
no     39922
yes     5289
Name: count, dtype: int64

In [114]:
for col in df.columns:
    print()
    print(str(col).upper())
    print(df[col].value_counts())


AGE
age
32    2085
31    1996
33    1972
34    1930
35    1894
      ... 
95       2
93       2
92       2
88       2
94       1
Name: count, Length: 77, dtype: int64

JOB
job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
unknown           288
Name: count, dtype: int64

MARITAL
marital
married     27214
single      12790
divorced     5207
Name: count, dtype: int64

EDUCATION
education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64

DEFAULT
default
no     44396
yes      815
Name: count, dtype: int64

BALANCE
balance
0        3514
1         195
2         156
4         139
3         134
         ... 
14204       1
8205        1
9710        1
7038        1
4416        1
Name: count, Length: 7168, dtype: int64

HOUSING
housing
yes    25130
no 

In [115]:
"""
CONVERT YES/NO TO BINARY 1/0
"""
def y_n_to_bool(value):
    if value == "yes":
        return 1
    else:
        return 0

binary_cols = ["default", "housing", "loan", "y"]

df[binary_cols] = df[binary_cols].map(y_n_to_bool)
df["default"].value_counts()

default
0    44396
1      815
Name: count, dtype: int64

In [116]:
"""
BALANCE THE DATASET
"""

class_0 = df[df["y"] == 0]
class_1 = df[df["y"] == 1]

class_0_downsampled = class_0.sample(n=len(class_1), random_state=1)

balanced_df = pd.concat([class_0_downsampled, class_1])
balanced_df = balanced_df.sample(frac=1, random_state=1).reset_index(drop=True)

df = balanced_df
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10578 entries, 0 to 10577
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        10578 non-null  int64 
 1   job        10578 non-null  object
 2   marital    10578 non-null  object
 3   education  10578 non-null  object
 4   default    10578 non-null  int64 
 5   balance    10578 non-null  int64 
 6   housing    10578 non-null  int64 
 7   loan       10578 non-null  int64 
 8   contact    10578 non-null  object
 9   day        10578 non-null  int64 
 10  month      10578 non-null  object
 11  duration   10578 non-null  int64 
 12  campaign   10578 non-null  int64 
 13  pdays      10578 non-null  int64 
 14  previous   10578 non-null  int64 
 15  poutcome   10578 non-null  object
 16  y          10578 non-null  int64 
dtypes: int64(11), object(6)
memory usage: 1.4+ MB


In [117]:
"""
CHECK IF CATEGORICAL COLUMNS
ARE ORDINAL OR NOMINAL
"""

catcols = ["job","marital","education","contact","month","poutcome"]

for col in catcols:
    print()
    print(str(col).upper())
    print(df.groupby(col)["y"].mean().sort_values(ascending=False))


JOB
job
student          0.732970
retired          0.693548
unemployed       0.570621
management       0.549177
self-employed    0.513736
admin.           0.498420
technician       0.486111
unknown          0.485714
housemaid        0.459916
services         0.440860
entrepreneur     0.430070
blue-collar      0.361963
Name: y, dtype: float64

MARITAL
marital
single      0.567359
divorced    0.512356
married     0.459626
Name: y, dtype: float64

EDUCATION
education
tertiary     0.578886
unknown      0.519588
secondary    0.470340
primary      0.411560
Name: y, dtype: float64

CONTACT
contact
cellular     0.572383
telephone    0.537931
unknown      0.238739
Name: y, dtype: float64

MONTH
month
sep    0.887789
dec    0.862069
mar    0.855172
oct    0.854497
apr    0.656428
feb    0.612500
aug    0.476125
nov    0.471345
jan    0.465574
jun    0.459209
jul    0.429452
may    0.350644
Name: y, dtype: float64

POUTCOME
poutcome
success    0.935885
other      0.627812
failure    0.529110
unk

In [118]:
"""
TURN CATEGORICAL DATA TO NUMERIC
FOR ORDINAL COLUMNS
"""

from sklearn.preprocessing import LabelEncoder

ordinal_cols = ["education", "marital", "contact", "poutcome"]

def ordinal_handler(df, cols):
    le = LabelEncoder()
    for col in cols:
        df[col] = le.fit_transform(df[col])
    return df

df = ordinal_handler(df, ordinal_cols)

In [119]:
"""
TURN CATEGORICAL DATA TO DUMMY
FOR NOMINAL COLUMNS
"""

from sklearn.preprocessing import OneHotEncoder

nominal_cols = ["month", "job"]

ohe = OneHotEncoder(sparse_output=False)

ohe_array = ohe.fit_transform(df[nominal_cols])
ohe_df = pd.DataFrame(ohe_array, columns=ohe.get_feature_names_out(nominal_cols))
df = pd.concat([df.drop(columns=nominal_cols), ohe_df], axis=1)

In [120]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10578 entries, 0 to 10577
Data columns (total 39 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                10578 non-null  int64  
 1   marital            10578 non-null  int64  
 2   education          10578 non-null  int64  
 3   default            10578 non-null  int64  
 4   balance            10578 non-null  int64  
 5   housing            10578 non-null  int64  
 6   loan               10578 non-null  int64  
 7   contact            10578 non-null  int64  
 8   day                10578 non-null  int64  
 9   duration           10578 non-null  int64  
 10  campaign           10578 non-null  int64  
 11  pdays              10578 non-null  int64  
 12  previous           10578 non-null  int64  
 13  poutcome           10578 non-null  int64  
 14  y                  10578 non-null  int64  
 15  month_apr          10578 non-null  float64
 16  month_aug          105

In [139]:
"""
PREP FEATURES AND TARGET
"""

X = df.drop(columns="y")
y = df["y"]

"""
SPLIT INTO TRAIN, TEST, VALIDATION
"""

from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=1
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=1
)

In [140]:
"""
TRAIN THE TREE
"""

from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=1),
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best F1 score:", grid.best_score_)


Best params: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 20}
Best F1 score: 0.8279588467660298


In [ ]:
"""
VALIDATE
"""

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier

best_tree = DecisionTreeClassifier(**grid.best_params_, random_state=1) # ** unpacks the dictionary
best_tree.fit(X_train, y_train)

y_val_pred = best_tree.predict(X_val)

print("Validation accuracy:\n", accuracy_score(y_val, y_val_pred))
print("Validation confusion matrix:\n", confusion_matrix(y_val, y_val_pred))
print("Validation classification report:\n", classification_report(y_val, y_val_pred))


Validation accuracy:
 0.8317272053372868
Validation confusion matrix:
 [[589 113]
 [114 533]]
Validation classification report:
               precision    recall  f1-score   support

           0       0.84      0.84      0.84       702
           1       0.83      0.82      0.82       647

    accuracy                           0.83      1349
   macro avg       0.83      0.83      0.83      1349
weighted avg       0.83      0.83      0.83      1349



In [142]:
"""
FINAL TEST
"""

y_test_pred = best_tree.predict(X_test)

print("Test accuracy:\n", accuracy_score(y_test, y_test_pred))
print("Test confusion matrix:\n", confusion_matrix(y_test, y_test_pred))
print("Test classification report:\n", classification_report(y_test, y_test_pred))

Test accuracy:
 0.8229363579080026
Test confusion matrix:
 [[645 151]
 [130 661]]
Test classification report:
               precision    recall  f1-score   support

           0       0.83      0.81      0.82       796
           1       0.81      0.84      0.82       791

    accuracy                           0.82      1587
   macro avg       0.82      0.82      0.82      1587
weighted avg       0.82      0.82      0.82      1587

